# Serving Frameworks

দুটি discrete-event simulation:

1. **STATIC বনাম CONTINUOUS batching।** STATIC (naive স্কিম): `N` টি requests-এর একটি batch গঠন করো, batch-এর সবচেয়ে **ধীর** সদস্য শেষ না হওয়া পর্যন্ত প্রতিটি slot চালাও, তবেই পরের `N` টি অপেক্ষমাণ requests-কে ভর্তি করো। যেসব slot-এর request আগে শেষ হয়ে যায় সেগুলো সেই batch-এর বাকি lifetime-এর জন্য idle পড়ে থাকে। CONTINUOUS ("in-flight") batching — যেমন Orca জনপ্রিয় করেছে এবং Hugging Face TGI ব্যবহার করে: `N` টি concurrent slots বজায় রাখো; যেই মুহূর্তে কোনো slot-এর request শেষ হয়, তা-ই পরের অপেক্ষমাণ request দিয়ে backfill করো। কোনো slot কখনো batch-এর বাকিদের জন্য অপেক্ষা করে না। উভয় কৌশল **অভিন্ন** workload প্রসেস করে, তাই তুলনাটি batching policy-র প্রভাবকেই বিচ্ছিন্ন করে।
2. **ATOMIC বনাম CHUNKED prefill** (README section 5)। একটি দীর্ঘ prompt-এর prefill-কে একটি ATOMIC ধাপ হিসেবে dispatch করা যায় যা prefill-এর *সম্পূর্ণ* সময়কালের জন্য অন্য প্রতিটি in-flight request-এর পরবর্তী decode ধাপকে আটকে রাখে (head-of-line blocking), অথবা ছোট CHUNKS-এ ভাগ করে সবার decode ধাপের সাথে interleave করা যায়, অন্য requests-এর দেখা worst-case বিলম্বকে পুরো prefill-এর বদলে এক চunk-এর সময়কালে সীমিত করে — chunked request-টি নিজের জন্য একটি ছোট, বাস্তব overhead-এর খরচে।

এটি Python-এর standard library-র উপর একটি বিশুদ্ধ discrete-event simulation — এটি সেই SCHEDULING সমস্যাকে মডেল করে যা vLLM/TGI/SGLang/TensorRT-LLM-শৈলীর serving frameworks সমাধান করে, কোনো বাস্তব transformer forward pass নয়।

**চালানোর নিয়ম:** উপরের দিক থেকে নিচের দিকে প্রতিটি cell চালান (runtime এক সেকেন্ডেরও কম)। প্রতিটি সেকশনের demo তার নিজ cell-এর শেষেই চলে।

In [ ]:
import random

random.seed(0)

NUM_SLOTS = 8          # accelerator একসাথে কতগুলো sequence প্রসেস করতে পারে
NUM_REQUESTS = 48       # পুরো সিমুলেটেড run জুড়ে serve করার মোট requests


def make_workload(num_requests):
    """Variable, skewed generation-length বিতরণ আঁকে -- বেশিরভাগ requests ছোট
    (একটি দ্রুত উত্তর), সংখ্যালঘু অনেক লম্বা (একটি বিস্তারিত ব্যাখ্যা)। এটি বাস্তব
    chat/completion traffic-কে প্রতিফলিত করে এবং ঠিক সেই আকৃতির workload যেখানে
    static batching সবচেয়ে বেশি ক্ষতিগ্রস্ত হয়।"""
    lengths = []
    for _ in range(num_requests):
        if random.random() < 0.75:
            lengths.append(random.randint(2, 8))     # short request
        else:
            lengths.append(random.randint(30, 50))   # long request
    return lengths

## 1. Static বনাম continuous batching: দুই পলিসি, হাতে ট্রেসযোগ্য উদাহরণ-সহ

প্রথমে দুটি batching পলিসি বাস্তবায়ন করা হয়। **Static batching** request queue-কে `num_slots`-এর নির্দিষ্ট-আকারের batches-এ কেটে দেয়; প্রতিটি batch তার সবচেয়ে ধীর সদস্যের যতগুলো ধাপ দরকার ততগুলোই চলে। **Continuous batching** `num_slots` টি concurrent slots বজায় রাখে; প্রতি simulated tick-এ একটি slot-এর request শেষ হলে তা পরের টিক-এর আগেই queue থেকে backfill হয়। তারপর একটি ছোট, হাতে ট্রেসযোগ্য উদাহরণে (4 requests, 2 slots) দুটোর আচরণ ধাপে ধাপে দেখানো হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Static batching simulation
# ---------------------------------------------------------------------------

def simulate_static_batching(gen_lengths, num_slots):
    """Request queue-কে `num_slots` আকারের নির্দিষ্ট-আকারের batches-এ কেটে দেয়।
    প্রতিটি batch তার সবচেয়ে ধীর সদস্যের যতগুলো ধাপ দরকার ততগুলো চলে; সেই batch-এর
    বাকি প্রতিটি slot নিজের request শেষ হয়ে গেলে idle পড়ে থাকে।"""
    total_time = 0
    total_slot_ticks = 0
    busy_slot_ticks = 0
    num_batches = 0

    queue = list(gen_lengths)
    while queue:
        batch = queue[:num_slots]
        queue = queue[num_slots:]
        num_batches += 1

        batch_duration = max(batch)          # the whole batch waits for the slowest member
        total_time += batch_duration
        total_slot_ticks += len(batch) * batch_duration
        busy_slot_ticks += sum(batch)        # real work done: each request's own length

    return {
        "total_time": total_time,
        "total_slot_ticks": total_slot_ticks,
        "busy_slot_ticks": busy_slot_ticks,
        "num_batches": num_batches,
    }


# ---------------------------------------------------------------------------
# 2. Continuous (in-flight) batching simulation
# ---------------------------------------------------------------------------

def simulate_continuous_batching(gen_lengths, num_slots):
    """`num_slots` টি concurrent slots বজায় রাখে। প্রতি simulated tick-এ, একটি
    request ধারণকারী প্রতিটি slot এক ধাপ কাজ করে; যার request মাত্র শেষ হলো সেই slot
    পরের টিক-এর আগে অপেক্ষমাণ queue থেকে অবিলম্বে backfill হয়, তাই কাজ উপলব্ধ থাকলে
    এটি কখনো idle পড়ে থাকে না।"""
    queue = list(gen_lengths)
    slots = [None] * num_slots   # প্রতিটি এন্ট্রি: সেই slot-এর request-এর অবশিষ্ট ধাপ, বা None

    def refill_empty_slots():
        for i in range(num_slots):
            if slots[i] is None and queue:
                slots[i] = queue.pop(0)

    refill_empty_slots()

    total_time = 0
    total_slot_ticks = 0
    busy_slot_ticks = 0

    while any(s is not None for s in slots) or queue:
        total_time += 1
        for i in range(num_slots):
            total_slot_ticks += 1          # every slot is "available capacity" every tick
            if slots[i] is not None:
                busy_slot_ticks += 1       # this slot did real work this tick
                slots[i] -= 1
                if slots[i] == 0:
                    slots[i] = None
        refill_empty_slots()               # backfill any slot that just freed up, immediately

    return {
        "total_time": total_time,
        "total_slot_ticks": total_slot_ticks,
        "busy_slot_ticks": busy_slot_ticks,
    }


# ---------------------------------------------------------------------------
# 3. ছোট, হাতে ট্রেসযোগ্য উদাহরণ প্রথম
# ---------------------------------------------------------------------------

def small_worked_example():
    print("=" * 70)
    print("1. A SMALL, HAND-TRACEABLE EXAMPLE")
    print("=" * 70)
    tiny_lengths = [10, 2, 15, 3]
    num_slots = 2
    print(f"4 requests needing {tiny_lengths} generation steps, {num_slots} slots\n")

    static_result = simulate_static_batching(tiny_lengths, num_slots)
    cont_result = simulate_continuous_batching(tiny_lengths, num_slots)

    print("Static batching: batch 1 = requests needing [10, 2] steps -> both slots")
    print("  occupied for 10 steps (slot 2 idle for 8 of them, since its request")
    print("  finished after step 2 but the batch can't turn over yet).")
    print("  batch 2 = requests needing [15, 3] steps -> both slots occupied for 15")
    print("  steps (slot 2 idle for 12 of them).")
    print(f"  -> total time = {static_result['total_time']} steps "
          f"(10 + 15), num_batches = {static_result['num_batches']}")

    print("\nContinuous batching: slot 2 finishes its 2-step request at time 2 and is")
    print("  immediately handed the 3rd request (15 steps) without waiting for slot 1's")
    print("  10-step request to finish; slot 1 then picks up the last request (3 steps)")
    print("  the moment it frees up at time 10.")
    print(f"  -> total time = {cont_result['total_time']} steps")

    print(f"\n-> Same 4 requests, same total work (30 generation steps), but static")
    print(f"   batching takes {static_result['total_time']} steps end-to-end while")
    print(f"   continuous batching takes only {cont_result['total_time']} steps --")
    print(f"   purely from never leaving a slot idle while work is waiting.")


small_worked_example()

## 2. বাস্তব-আকৃতির workload-এ পূর্ণ তুলনা

একটি skewed বিতরণ (75% ছোট: 2-8 ধাপ, 25% লম্বা: 30-50 ধাপ) থেকে আঁকা 48 টি requests, 8 টি concurrent slots-এ, **অভিন্ন** workload-এ static বনাম continuous batching সম্পূর্ণ তুলনা করা হয় — মোট সময়, batch-সংখ্যা, slot utilization — এবং throughput লাভ সরাসরি মাপা হয়।

In [ ]:
# ---------------------------------------------------------------------------
# 4. বাস্তব-আকৃতির workload-এ পূর্ণ তুলনা
# ---------------------------------------------------------------------------

def full_comparison():
    print("\n" + "=" * 70)
    print("2. FULL WORKLOAD: STATIC vs. CONTINUOUS BATCHING")
    print("=" * 70)

    gen_lengths = make_workload(NUM_REQUESTS)
    total_work = sum(gen_lengths)
    print(f"{NUM_REQUESTS} requests, {NUM_SLOTS} concurrent slots, generation lengths drawn")
    print("from a skewed distribution (75% short: 2-8 steps, 25% long: 30-50 steps).")
    print(f"Total real work across all requests: {total_work} generation-steps.\n")

    static_result = simulate_static_batching(gen_lengths, NUM_SLOTS)
    cont_result = simulate_continuous_batching(gen_lengths, NUM_SLOTS)

    static_util = static_result["busy_slot_ticks"] / static_result["total_slot_ticks"]
    cont_util = cont_result["busy_slot_ticks"] / cont_result["total_slot_ticks"]

    print(f"{'strategy':<22}{'total time (steps)':>20}{'slot-ticks (busy/total)':>26}{'utilization':>14}")
    print(f"{'Static batching':<22}{static_result['total_time']:>20}"
          f"{str(static_result['busy_slot_ticks']) + '/' + str(static_result['total_slot_ticks']):>26}"
          f"{static_util:>13.1%}")
    print(f"{'Continuous batching':<22}{cont_result['total_time']:>20}"
          f"{str(cont_result['busy_slot_ticks']) + '/' + str(cont_result['total_slot_ticks']):>26}"
          f"{cont_util:>13.1%}")

    speedup = static_result["total_time"] / cont_result["total_time"]
    util_gain = cont_util / static_util

    print(f"\n-> Both strategies perform the exact same {total_work} generation-steps")
    print(f"   of real work, in the same arrival order, on the same {NUM_SLOTS} slots.")
    print(f"   Static batching needed {static_result['total_time']} total time steps and formed")
    print(f"   {static_result['num_batches']} batches, achieving only {static_util:.1%} slot utilization --")
    print(f"   most of the waste comes from short requests sharing a batch with a long")
    print(f"   one and then sitting idle until the whole batch turns over.")
    print(f"   Continuous batching cleared the identical workload in {cont_result['total_time']} time")
    print(f"   steps ({speedup:.2f}x faster) at {cont_util:.1%} utilization ({util_gain:.2f}x higher) by")
    print(f"   backfilling a freed slot immediately instead of waiting for the batch.")
    print(f"   This is exactly the throughput gain Orca/Hugging Face TGI's continuous")
    print(f"   batching delivers over naive static batching on real, variable-length")
    print(f"   chat traffic -- no change to the model, only to the scheduling policy.")


full_comparison()

## 3. Chunked prefill বনাম atomic prefill: head-of-line blocking, মাপা

একটি continuous-batching server-এ যখন একটি নতুন দীর্ঘ prompt আসে (এখানে 2000-token prefill), ATOMIC policy তা একটি অবিরাম ধাপে চালায় — অন্য প্রতিটি in-flight request-এর পরবর্তী decode ধাপ prefill-এর *সম্পূর্ণ* দৈর্ঘ্য পর্যন্ত বিলম্বিত হয়। CHUNKED policy prefill-কে ছোট chunks-এ ভাগ করে (1000 → 64 token) প্রতিটি chunk-boundary-তে একটি ছোট স্থির overhead-এর খরচে। Demo-টি বিভিন্ন chunk size-এর জন্য অন্য-স্লট worst-case delay এবং chunked request-টির নিজের মোট টিক সংখ্যা মাপে।

In [ ]:
# ---------------------------------------------------------------------------
# 5. Chunked prefill বনাম atomic prefill: head-of-line blocking, মাপা
# (README section 5)
# ---------------------------------------------------------------------------

PREFILL_LEN = 2000          # এইমাত্র আসা দীর্ঘ prompt-এর tokens
CHUNK_OVERHEAD = 2           # প্রতি chunk-এ ছোট স্থির dispatch/resume খরচ (ticks)


def simulate_prefill_policy(prefill_len, chunk_size, chunk_overhead):
    """ফেরত দেয় (worst_case_other_slot_delay, total_ticks_to_finish_prefill).

    chunk_size == prefill_len হলে ATOMIC policy মডেল হয় (একটি বিশাল "chunk"):
    অন্য slots তাদের পরবর্তী decode step-এর আগে FULL prefill length অপেক্ষা করে,
    এবং কোনো chunking overhead দিতে হয় না। chunk_size < prefill_len হলে
    CHUNKED prefill মডেল হয়: অন্য slots সর্বোচ্চ একটি chunk's worth টিক অপেক্ষা
    করে, কিন্তু interruptible হওয়ার সুবিধার জন্য request প্রতিটি chunk
    boundary-তে একটি ছোট স্থির overhead দেয়।"""
    num_chunks = -(-prefill_len // chunk_size)   # ceiling division
    worst_case_other_slot_delay = min(chunk_size, prefill_len)
    if num_chunks == 1:
        total_ticks = prefill_len                 # atomic: no chunk boundaries at all
    else:
        total_ticks = prefill_len + num_chunks * chunk_overhead
    return worst_case_other_slot_delay, total_ticks, num_chunks


def chunked_prefill_demo():
    print("\n" + "=" * 70)
    print("3. HEAD-OF-LINE BLOCKING: ATOMIC vs. CHUNKED PREFILL")
    print("=" * 70)
    print(f"Setup: several requests are already mid-decode in a continuous-batching")
    print(f"server (README section 4) when ONE new request arrives needing a")
    print(f"{PREFILL_LEN}-token prefill (a long document or long system prompt).\n")

    atomic_delay, atomic_total, _ = simulate_prefill_policy(PREFILL_LEN, PREFILL_LEN, CHUNK_OVERHEAD)
    print(f"ATOMIC prefill (current naive behavior): the {PREFILL_LEN}-token prefill runs")
    print(f"as one uninterruptible step.")
    print(f"  -> every OTHER in-flight request's next decode step is delayed by "
          f"{atomic_delay} ticks")
    print(f"  -> the long-prefill request itself finishes prefill after {atomic_total} ticks\n")

    print(f"{'chunk size':>12}{'num chunks':>13}{'other-slot worst delay':>26}{'chunked total ticks':>22}{'overhead vs atomic':>21}")
    chunk_sizes = [1000, 500, 256, 128, 64]
    results = []
    for chunk_size in chunk_sizes:
        delay, total, num_chunks = simulate_prefill_policy(PREFILL_LEN, chunk_size, CHUNK_OVERHEAD)
        overhead_pct = (total - atomic_total) / atomic_total
        results.append((chunk_size, num_chunks, delay, total, overhead_pct))
        print(f"{chunk_size:>12}{num_chunks:>13}{delay:>26}{total:>22}{overhead_pct:>20.1%}")

    best_practical = results[2]   # chunk_size=256, a realistic real-system choice
    chunk_size, num_chunks, delay, total, overhead_pct = best_practical
    speedup_in_worst_case_delay = atomic_delay / delay

    print(f"\n-> With chunk_size={chunk_size} (a realistic real-system choice, {num_chunks} chunks):")
    print(f"   other in-flight requests' worst-case delay drops from {atomic_delay} ticks (atomic)")
    print(f"   to just {delay} ticks -- a {speedup_in_worst_case_delay:.1f}x reduction in the worst latency")
    print(f"   spike anyone else in the batch experiences. The long-prefill request")
    print(f"   itself pays a small, REAL, honestly-measured price for being")
    print(f"   interruptible: {total} ticks to finish its own prefill instead of")
    print(f"   {atomic_total}, only {overhead_pct:.1%} slower end-to-end for itself.")
    print(f"   Smaller chunks push the worst-case delay down further but cost more")
    print(f"   total overhead (see the table) -- chunk size is a real, tunable knob")
    print(f"   trading other requests' latency against this request's own throughput,")
    print(f"   exactly the parameter real chunked-prefill schedulers expose.")


chunked_prefill_demo()

In [ ]:
def main():
    small_worked_example()
    full_comparison()
    chunked_prefill_demo()


main()